# Top20_2_Code — Top20 PDF parser + market commentary → llmwiki

Version 2 of `Top20-Ubuntu.ipynb`. The per-stock parsing (pages 3+ → `Trading.Stock_Options`)
is **unchanged** — those cells are copied verbatim. The one new feature mirrors
`ETF_2_Code.ipynb` / `playbook_etf.py` (Feature A in `docs/Design-Plan.md`):

- The Top20's printed **page 1** (physical page 2 — physical page 1 is the Disclosure
  Statement) carries the free-text market commentary (`"Market Expectations for <month>: ..."`
  plus `"Notes on current edition: ..."`). It is captured as `commentary_text` while the PDF is
  read, saved next to the PDF as `<name>_commentary.txt`, and sent to llmwiki via
  `POST {LLMWIKI_BASE_URL}/ingest` as `{text, title}` with the same retry-then-skip policy the
  ETF pipeline uses (`MAX_LLMWIKI_RETRY`, then log and move on — a llmwiki outage never aborts
  the run). If `LLMWIKI_BASE_URL` is unset the upload is skipped with a log line.

Run the cells top-to-bottom; update `InputDate` (and the `file` path for your host) first.


In [ ]:
# !pip install pdfplumber
# !pip install PyMuPDF

In [ ]:
#Importing required libraries
import traceback
import pandas as pd
import numpy as np
import re
import pdfplumber
from collections import namedtuple
from calendar import month_name
import fitz
import os
import time
import logging
import requests
from dotenv import load_dotenv
import dataUtil as DU


In [ ]:
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

## Locate & read the source PDF

Same convention as `Top20-Ubuntu.ipynb`: several candidate paths, the last uncommented
assignment wins. Physical page 1 (Disclosure Statement) is skipped; physical page 2 (printed
"Page 1") is only used for the commentary; pages 3+ feed the per-stock parser exactly as before.


In [ ]:
#Import PDF file from path
InputDate = '2025-07-21'
currentLCR = f"912 Top20 {InputDate}.pdf"
# Windows / WSL candidates (Google Drive for desktop), same as Top20-Ubuntu.ipynb:
# file=f"I:\\My Drive\\ReadProjects\\Neural Matrix Investment\\LCR\\{currentLCR}"
# file=f"/mnt/i/My Drive/ReadProjects/Neural Matrix Investment/LCR/Top20s/{currentLCR}"
# Pure Ubuntu host: rclone mirror (see README.md -> "Getting the PDFs")
file = os.path.expanduser(f"~/lcr/prod/top20/{currentLCR}")
print('Processing LCR:', file)


In [ ]:
# --- new in v2: capture the page-1 market commentary while reading the PDF ---
# Physical page 2 is the printed "Page 1": masthead, then "Market Expectations for <month>: ..."
# and "Notes on current edition: ...", then the footer "<date>  912 Top 20  Vol N Issue N Page 1".
# The commentary is the block from the "Market Expectations" line up to (not including) that
# footer line. Confirmed against 912 Top20 2025-07-21.pdf and LCR Top20 2022-11-21.pdf.
COMMENTARY_START_MARKER = "Market Expectations"
COMMENTARY_END_PATTERN = re.compile(r"Vol\s+\d+\s+Issue\s+\d+")   # page footer line

def split_commentary(page_text: str) -> str:
    lines = page_text.split("\n")
    start = next((i for i, l in enumerate(lines)
                  if l.strip().startswith(COMMENTARY_START_MARKER)), None)
    if start is None:
        logging.warning(f"'{COMMENTARY_START_MARKER}' not found on page 2; no commentary captured.")
        return ""
    end = next((i for i in range(start + 1, len(lines))
                if COMMENTARY_END_PATTERN.search(lines[i])), len(lines))
    return "\n".join(l.rstrip() for l in lines[start:end]).strip()

commentary_text = ""
with fitz.open(file) as doc:
    text = ""
    text_full = ""
    c=0
    for page in doc:
        c += 1
        if c == 2:
            # printed page 1: commentary only, never fed to the stock parser
            commentary_text = split_commentary(page.get_text())
            continue
        if c < 3: continue   # physical page 1 = Disclosure Statement
        text = page.get_text()
        text_full += text
print("==> market commentary:\n", commentary_text)
print(text_full)


In [ ]:
iterates = iter(text_full.split('\n'))
data_foo = [['','','','','']]

#Basic columns to be read from PDF
col= ["Symbol", "Status", "Expiration", "PnC", "Price" , "Entry1", "Entry2", "Target", "Stop"]
data_f = []
for x in col:
    data_f.append('')
data_foo = [data_f]
df = pd.DataFrame(data_foo,columns=col)

In [ ]:
# v2: row values are written with df.loc[row, col] = ... instead of df[col][row] = ...
# (chained assignment is a silent no-op on pandas >= 2 Copy-on-Write / pandas 3, which
# the current .venv uses). Parsing logic is otherwise identical to Top20-Ubuntu.ipynb.
Stock_name_tmp = None
Status = None
line_num = -1
row_number = -1
for line in iterates:
        line_num += 1
        print("line {}> {}".format(line_num, line))
        if line.strip() == '':
            continue
        if re.search("\[\w*\]", line):
            Stock_name_tmp = (re.findall(r"\[\w*\]", line)[0][1:-1])
            print("{},{}> find stock name: {}".format(line_num, row_number, Stock_name_tmp))
            Status = re.findall(r"\S*", next(iterates))[0]
        else:
            if (line.find("Aggressive Traders:") > -1):
                pass
            elif (line.find("Entry @ ") > -1) or (line.find("range of") > -1):
                df.loc[len(df.index)] = data_f
                row_number+= 1
                df.loc[row_number, 'Symbol'] = Stock_name_tmp
                df.loc[row_number, 'Status'] = Status
                entry = line.split('@')[1]
                mtch_entry = re.finditer(r"\d+(\.)?\d+", entry)
                ent = []
                for n in mtch_entry:
                    ent.append(n[0])
                ent1 = ent[0]
                ent2 = ent[1]
                df.loc[row_number, "Entry1"] = ent1
                df.loc[row_number, "Entry2"] = ent2
            elif (line.find("Target @ ") > -1):
                target = line.split('@')[1]
                mtch_trgt = re.finditer(r"\d+(\.)?\d+", target)
                trgt = []
                for z in mtch_trgt:
                    trgt.append(z[0])
                df.loc[row_number, "Target"] = trgt[0]
            elif ((line.find("STO") > -1) or (line.find("BTO") > -1)):
                decission = re.findall(r"Put|Call", line)[0][0]
                print("{}> {} from line={} found PnC={}".format(row_number, df['Symbol'][row_number], line, decission))
                exp_date = re.findall(r"\(...*\)", line)[0][1:-4]
                df.loc[row_number, 'PnC'] = decission
                df.loc[row_number, 'Expiration'] = exp_date
                price = line.split('(')[0]
                matches = re.finditer(r"\d+(\.)?\d+", price)
                tmp_price = []
                for m in matches:
                    tmp_price.append(m[0])
                prc = tmp_price[0]
                df.loc[row_number, "Price"] = prc
            elif (line.find("Stop") > -1):
                stop = line.split('@')[1]
                mtch_stop = re.finditer(r"\d+(\.)?\d+", stop)
                stp = []
                for q in mtch_stop:
                    stp.append(q[0])
                df.loc[row_number, "Stop"] = stp[0]
            else:
                pass
print(df)

In [ ]:
df = df[:-1]
df = df.fillna('')
df['Entry_Sign'] = df['PnC'].apply(lambda x: '<' if x == 'P' else '>')
df['Target_Sign'] = df['PnC'].apply(lambda x: '>' if x == 'P' else '<')
df['Stop_Sign'] = df['Entry_Sign']

In [ ]:
df

In [ ]:
df["Entry1"], df["Entry2"] = np.where(df["PnC"]=="C", [df["Entry2"], df["Entry1"]], [df["Entry1"], df["Entry2"] ])

In [ ]:
#Arranging columns as required in output file
df['Date'] = InputDate
df = df[["Date", "Symbol", "Status", "Expiration", "PnC", "Price" , "Entry_Sign", "Entry1", "Entry2", "Target_Sign", "Target", "Stop_Sign", "Stop"]]

df.rename(columns = {'Price':'Strike'}, inplace = True)
#Exports the final dataframe to txt file
df.to_csv(file.replace('.pdf', '.csv'), sep=',', index = None)

In [ ]:
from dotenv import load_dotenv
import logging
import numpy as np
import dataUtil as DU

In [ ]:
df.info()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Expiration"] = pd.to_datetime(df["Expiration"])
df = df.replace(r'^\s*$', np.nan, regex=True)
# final_df['L_Strike'] = final_df['L_Strike'].astype(float)
df = df.astype({'Strike':'float','Entry1':'float','Entry2':'float','Target':'float','Stop':'float'})
df.info()

In [ ]:
df

## Configuration

`DB_Config.env` is the repo convention (both notebooks and `playbook_etf.py` load it); it is also
where the `LLMWIKI_*` keys live — see `.env.example`. Falls back to `.env` if `DB_Config.env` is
not present on this host.


In [ ]:
if not load_dotenv("DB_Config.env"):   #Check path for env variables
    if load_dotenv(".env"):
        logging.warning("DB_Config.env not found; loaded .env instead.")
    else:
        logging.warning("Neither DB_Config.env nor .env found; relying on the process environment.")
logging.getLogger().setLevel(logging.DEBUG)


## Feature A — market commentary → llmwiki

Same logic as the ETF pipeline (`playbook_etf.py` → `upload_commentary`, which superseded the
`/upload` stub in `ETF_2_Code.ipynb`): save the commentary locally next to the PDF, then
`POST {LLMWIKI_BASE_URL}/ingest` with `{"title": ..., "text": "Date: <InputDate>\n\n<commentary>"}`
and a bearer token. Retries `MAX_LLMWIKI_RETRY` times with a short backoff, then logs and skips —
it never raises, so a llmwiki outage does not stop the DB load below. Set `SKIP_LLMWIKI = True`
to parse/save only.


In [ ]:
SKIP_LLMWIKI = False

def upload_commentary_to_llmwiki(text: str, date_str: str) -> bool:
    """POST the market commentary to llmwiki ``/ingest``; retry then log-and-skip, never raise."""
    base_url = os.environ.get("LLMWIKI_BASE_URL")
    if not base_url:
        logging.info(f"LLMWIKI_BASE_URL not set; skipping llmwiki upload for {date_str}.")
        return False
    token = os.environ.get("LLMWIKI_API_TOKEN", "")
    max_retry = int(os.environ.get("MAX_LLMWIKI_RETRY", "5"))
    timeout = float(os.environ.get("LLMWIKI_TIMEOUT", "30"))

    url = f"{base_url.rstrip('/')}/ingest"
    title = f"LCR Top20 Market Commentary {date_str}"
    payload = {"title": title, "text": f"Date: {date_str}\n\n{text}"}
    headers = {"Authorization": f"Bearer {token}"}
    for attempt in range(1, max_retry + 1):
        try:
            resp = requests.post(url, json=payload, headers=headers, timeout=timeout)
            resp.raise_for_status()
            logging.info(f"llmwiki ingest succeeded for {date_str} on attempt {attempt}: {resp.text}")
            return True
        except Exception as e:
            logging.error(f"llmwiki ingest attempt {attempt}/{max_retry} failed for {date_str}: {e}",
                          exc_info=(attempt == max_retry))
            if attempt < max_retry:
                time.sleep(min(2 * attempt, 10))
    logging.error(f"llmwiki ingest failed after {max_retry} attempts for {date_str}; skipping.")
    return False

commentary_path = file.replace('.pdf', '_commentary.txt')
if commentary_text:
    with open(commentary_path, 'w') as f:
        f.write(commentary_text)
    print(f"Saved market commentary to {commentary_path}")
    if SKIP_LLMWIKI:
        print("SKIP_LLMWIKI=True — not sending commentary to llmwiki.")
    else:
        upload_commentary_to_llmwiki(commentary_text, InputDate)
else:
    logging.warning("No market commentary text captured from page 2 (printed page 1).")


## Load to MySQL

Unchanged from `Top20-Ubuntu.ipynb`: appends to `Trading.Stock_Options` (no dedupe — re-running
the same date appends duplicate rows).


In [ ]:
opt_tbl = "Stock_Options_v1"
trd_DB = "Trading"
DU.StoreEOD(df, trd_DB, opt_tbl)